# Auditing data before you trust it

_Doing ML for Real — the skills that matter_

**Most model failures are data failures. Audit the data first, or your metrics are lying to you.**

A dataset is a promise about the world: each column means something, in some unit, within some range. An audit is how you check the promise before you bet a model on it.

       The audit makes vague worry concrete. Instead of "the data looks off", you produce numbers: this column is 12% null, those 30 rows are duplicates, that feature drifted with a PSI (Population Stability Index) of 0.4.

---

This notebook builds a data audit from scratch, one check at a time. Run each cell, read the note above it, then tackle the practice problems at the end. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q pandera ydata-profiling
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — pandera + ydata-profiling + scipy

A full audit has three independent checks. We build them one at a time: (1) a **schema contract** that rejects bad rows, (2) a **missingness + duplicate** report, (3) a **drift** check (PSI) comparing a current sample against a reference. _The reference code below is illustrative — it assumes audit frames named `df`, `reference_df`, and `current_df`; wire in your own DataFrames to run it._

### Step 1 — Declare and enforce a schema contract

A schema makes each column's promise explicit: its type, its allowed range or categories, whether nulls are permitted. `schema.validate(...)` then *rejects* rows that break the contract instead of letting them flow on silently. `lazy=True` collects every failure in one pass; `strict=True` also rejects any unexpected extra column.

In [ ]:
import numpy as np
import pandas as pd
import pandera as pa
from pandera import Column, Check, DataFrameSchema
from scipy import stats

# ---------------------------------------------------------------
# 1. SCHEMA CONTRACT  -- declare type, range, allowed categories,
#    then validate. Bad rows raise instead of silently flowing on.
# ---------------------------------------------------------------
schema = DataFrameSchema({
    "age":      Column(int,   Check.in_range(0, 120)),
    "income":   Column(float, Check.ge(0), nullable=True),     # nulls allowed, negatives not
    "segment":  Column(str,   Check.isin(["free", "pro", "enterprise"])),
    "country":  Column(str,   Check.str_length(2, 2)),         # ISO-2 codes
    "label":    Column(int,   Check.isin([0, 1])),
}, strict=True)            # strict=True -> reject unexpected columns too

validated = schema.validate(df, lazy=True)   # lazy=True collects ALL failures, not just the first

### Step 2 — Report missingness and duplicates

Two quiet data killers. First, the **null rate** per column — and crucially whether missingness correlates with the label, a hint that the data is MNAR/MAR rather than missing at random. Second, **duplicates**: exact full-row copies, and repeated entity keys (the same `user_id` twice). The `ydata-profiling` report bundles all of this into one HTML file.

In [ ]:
# ---------------------------------------------------------------
# 2. MISSINGNESS + DUPLICATE REPORT
# ---------------------------------------------------------------
missing = df.isna().mean().sort_values(ascending=False)        # null rate per column
print("null rate per column:\n", missing)

# is missingness tied to the label?  (a hint of MNAR / MAR, not MCAR)
for c in df.columns:
    if df[c].isna().any():
        rate_by_label = df.assign(_miss=df[c].isna()).groupby("label")["_miss"].mean()
        print(c, "missing-rate by label:\n", rate_by_label)

exact_dupes = df.duplicated().sum()                            # full-row duplicates
key_dupes   = df.duplicated(subset=["user_id"]).sum()          # same entity repeated
print("exact dupes:", exact_dupes, " duplicate user_ids:", key_dupes)

# full automated profile (HTML) -- types, missingness, dupes, correlations, alerts
from ydata_profiling import ProfileReport
ProfileReport(df, title="Data audit", minimal=True).to_file("audit.html")

### Step 3 — Measure drift with PSI (and a KS test)

The **Population Stability Index** quantifies how much a feature's distribution shifted from a reference to a current sample: `PSI = sum (a - e) * ln(a / e)` over quantile bins of the reference. Rules of thumb: `<0.1` stable, `<0.25` watch, above that real drift. The Kolmogorov-Smirnov test is a complementary two-sample check on the raw distributions.

In [ ]:
# ---------------------------------------------------------------
# 3. POPULATION STABILITY INDEX  -- drift of CURRENT vs REFERENCE
#    PSI = sum_i (a_i - e_i) * ln(a_i / e_i)
# ---------------------------------------------------------------
def psi(reference, current, bins=10):
    edges = np.quantile(reference, np.linspace(0, 1, bins + 1))  # quantile bins from reference
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(reference, bins=edges)[0] / len(reference)  # expected (reference) shares
    a = np.histogram(current,   bins=edges)[0] / len(current)    # actual (current) shares
    e = np.clip(e, 1e-6, None); a = np.clip(a, 1e-6, None)       # avoid log(0)
    return float(np.sum((a - e) * np.log(a / e)))

for col in ["age", "income"]:
    score = psi(reference_df[col], current_df[col])
    flag = "STABLE" if score < 0.1 else "WATCH" if score < 0.25 else "DRIFT!"
    print(f"{col}: PSI={score:.3f}  ({flag})")

# a complementary two-sample distribution test (KS = Kolmogorov-Smirnov)
ks = stats.ks_2samp(reference_df["income"].dropna(), current_df["income"].dropna())
print("income KS statistic:", ks.statistic, "p-value:", ks.pvalue)

## Visualize the data & results

_Split a real dataset into a reference half and a current half — which features drift, and by how much (PSI)?_ We build this in three steps: (1) split the wine data into reference vs current, (2) define the PSI function, (3) rank every feature by its drift.

### Step 1 — Split a real dataset into reference vs current

We use the 178-row wine dataset, whose rows are ordered by cultivar. Splitting it in half therefore creates a deliberate distribution shift between the first and second halves — a clean way to see PSI light up on real data.

In [ ]:
import numpy as np
from sklearn.datasets import load_wine

wine = load_wine()
X, names = wine.data, wine.feature_names          # 178 x 13 real chemical measurements
half = X.shape[0] // 2                             # rows are ordered by cultivar
ref, cur = X[:half], X[half:]                      # reference = first half, current = second half

### Step 2 — Define the PSI function

Same formula as the reference implementation: bin the reference by quantiles, compare the expected (reference) and actual (current) shares in each bin, and sum `(a - e) * ln(a / e)`. Clipping away from zero avoids `log(0)` when a bin is empty in one sample.

In [ ]:
def psi(reference, current, bins=10):
    edges = np.quantile(reference, np.linspace(0, 1, bins + 1))  # quantile bins from reference
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(reference, bins=edges)[0] / len(reference)  # expected shares
    a = np.histogram(current,   bins=edges)[0] / len(current)    # actual shares
    e = np.clip(e, 1e-6, None); a = np.clip(a, 1e-6, None)       # avoid log(0)
    return float(np.sum((a - e) * np.log(a / e)))                # PSI = sum (a-e) ln(a/e)

### Step 3 — Rank features by drift

Score every one of the 13 features, sort by descending PSI, and print the worst eight. The chemistry that separates the cultivars — proline, ash alkalinity, flavanoids — shows the biggest shift, exactly the columns a real audit would flag first.

In [ ]:
scores = [(names[i], round(psi(ref[:, i], cur[:, i]), 3)) for i in range(X.shape[1])]
scores.sort(key=lambda t: -t[1])
for name, s in scores[:8]:
    print(f"{name:30s} PSI={s}")
# proline    6.667 | alcalinity_of_ash 4.976 | flavanoids 2.827 | hue 2.742
# od280...   2.450 | total_phenols     2.267 | nonflavanoid_phenols 1.505 | malic_acid 1.419

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A teammate cleans a survey dataset by running df['income'] = df['income'].fillna(df['income'].mean()) as the very first step. The income column is 35% null. Why is this risky, and what should they have done first?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Ask WHY income is missing before filling it. — _High earners often decline to report income. If "missing" depends on the value itself, the missingness is MNAR (Missing Not At Random) and carries signal._
- Cross-tabulate the missingness flag against the label and other columns. — _If income-missing correlates with the target (or with wealth proxies), the nulls are informative, not random noise._
- Add a "was-missing" indicator column, THEN impute. — _The flag preserves the signal that a value was missing, so a mean-fill no longer erases it. Imputing blindly throws that signal away._

**Answer:** Mean-imputing first assumes the data is MCAR/MAR. With 35% nulls that likely correlate with the value (MNAR), it erases real signal. First profile the mechanism (cross-tab missingness vs the label), add a "was-missing" flag, then impute.

</details>

**Problem 2.** You ship a model that scored AUC 0.92 offline. In production it scores 0.74. The training data audit was clean. Using the data-audit playbook, where do you look first, and which tool do you use?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize this as a train/serving parity problem, not a training-data problem. — _The training audit was clean, so the defect is in the gap between training and serving data — the one slice that was never audited._
- Fix the training set as the reference distribution and compute PSI per feature on the live serving inputs. — _PSI (Population Stability Index) per feature surfaces exactly which features drifted; a value above 0.25 flags a big shift likely to break the model._
- For the top-PSI features, check units, encodings, and new null patterns. — _Real drift is often a renamed column, a unit change (dollars to cents), or a new missingness pattern in the serving pipeline._

**Answer:** It is a parity/drift failure. Use the training set as the reference and compute PSI per feature on the serving inputs; investigate the highest-PSI features for unit changes, encoding mismatches, or new null patterns. Audit serving data, not just training data.

</details>